[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_14_MCP_Model_Context_Protocol.ipynb)

# 🔌 Lesson 14: MCP — Model Context Protocol
## *The Universal Plug Standard for AI Tools*

---

### 🎯 What You'll Learn

By the end of this lesson, you'll understand:
- **Why MCP exists** — the problem it solves in the AI ecosystem
- **MCP architecture** — hosts, clients, servers, and the protocol flow
- **Building MCP servers** — tools, resources, and prompts using FastMCP (Python)
- **How Claude uses MCP** — the exact protocol messages exchanged
- **Capstone**: Build a mini `ResearchMCP` server with 3 real tools

---

### 🤔 Why This Lesson Matters

**Fun fact:** You've been *using* MCP this entire course. When you run Claude in Cowork mode and Claude can read your files, search the web, or create calendar events — that's MCP at work.

MCP is rapidly becoming the **USB-C of AI** — one standard that lets any AI model connect to any tool or data source. In 2025, it went from an Anthropic internal project to an open standard adopted by OpenAI, Google, Microsoft, and hundreds of tool providers.

If you want to build AI agents that are production-grade and pluggable, **you need to know MCP**.

## 🛠️ Setup

Run this first. It installs everything needed.

In [ ]:
# Install required packages
!pip install anthropic mcp fastmcp httpx -q

print("✅ Packages installed!")

In [ ]:
import os

# Load API key from Colab Secrets (Key icon in left sidebar → add ANTHROPIC_API_KEY)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Running locally
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"
    print("✅ API key loaded from environment")

---

## 📖 Part 1: The Problem MCP Solves

### The Fragmentation Problem

Before MCP, every AI system needed **custom integrations** for every tool:

```
Claude + Slack   → custom Claude-Slack connector
Claude + GitHub  → custom Claude-GitHub connector  
GPT + Slack      → custom GPT-Slack connector (different!)
GPT + GitHub     → custom GPT-GitHub connector (different!)
Gemini + Slack   → yet another custom connector...
```

With **N models** and **M tools**, you need **N×M custom integrations**. It was chaos.

### The MCP Solution

MCP defines one protocol that any AI model can speak and any tool can implement:

```
Claude  ──┐
GPT-4   ──┤  [MCP Protocol]  ──  Slack MCP Server
Gemini  ──┘                  ──  GitHub MCP Server
                             ──  Your Custom MCP Server
```

Now you only need **N + M** integrations (one per model, one per tool). Like USB-C: one plug standard, works everywhere.

### What MCP Defines

MCP standardizes three types of capabilities a server can expose:

| Capability | Description | Example |
|-----------|-------------|----------|
| **Tools** | Functions the AI can call (with side effects) | `search_web()`, `create_file()` |
| **Resources** | Read-only data sources | File contents, database rows |
| **Prompts** | Reusable prompt templates | Pre-built system prompts |

You've been using all three without knowing it! When Claude reads your files in Cowork → Resources. When it calls calendar tools → Tools. When it uses predefined role instructions → Prompts.

---

## 🏗️ Part 2: MCP Architecture — Hosts, Clients, Servers

MCP has three roles:

```
┌─────────────────────────────────────────────┐
│                    HOST                      │
│   (Claude Desktop, Cowork, your AI app)      │
│                                              │
│  ┌────────────────────────────────────────┐  │
│  │              MCP CLIENT                │  │
│  │  (manages connections to servers)      │  │
│  └──────┬──────────────┬─────────────────┘  │
└─────────┼──────────────┼─────────────────────┘
          │              │
          ▼              ▼
   ┌────────────┐  ┌────────────┐
   │ MCP SERVER │  │ MCP SERVER │
   │ (GitHub)   │  │ (Weather)  │
   └────────────┘  └────────────┘
```

### How a Tool Call Works (Protocol Flow)

1. **Claude decides** it needs to call a tool (e.g., `search_web`)
2. **Host/Client** routes the call to the correct MCP server
3. **MCP Server** executes the function and returns result
4. **Host** injects the result back into Claude's context
5. **Claude** continues generating with that info

### Transport Options

MCP servers can communicate via:
- **stdio** — stdin/stdout pipes (local subprocess, most common)
- **SSE (Server-Sent Events)** — HTTP streaming (remote server)
- **Streamable HTTP** — newer standard for remote servers

In this lesson, we'll use stdio for local development and in-process testing for Colab compatibility.

---

## 🚀 Part 3: Building Your First MCP Server with FastMCP

FastMCP is the high-level Python framework for building MCP servers. Think of it like FastAPI — decorator-based, minimal boilerplate.

### 3.1 The Simplest Possible MCP Server

In [ ]:
# The anatomy of a FastMCP server
# (We define it as a Python string so we can inspect it and also save it as a .py file)

SIMPLE_SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP

# 1. Create the server (give it a name)
mcp = FastMCP("MyFirstServer")

# 2. Decorate functions to expose them as TOOLS
@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together. Returns the sum."""
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    """Generate a friendly greeting for the given name."""
    return f"Hello, {name}! Welcome to MCP."

# 3. Run the server (stdio transport by default)
if __name__ == "__main__":
    mcp.run()
'''

print(SIMPLE_SERVER_CODE)

# Save to disk so we can reference it
with open("/tmp/simple_mcp_server.py", "w") as f:
    f.write(SIMPLE_SERVER_CODE.strip())

print("✅ Server code saved to /tmp/simple_mcp_server.py")

### 🔑 Key Concepts

**Three things make a function into an MCP tool:**

1. **`@mcp.tool()` decorator** — registers the function with the server
2. **Type annotations** (`a: int`, `b: int`) — MCP uses these to generate the JSON schema that Claude reads to know what arguments to pass
3. **Docstring** — becomes the tool's description that Claude reads to decide *when* to use this tool

The docstring is your **prompt to Claude** for this tool. Write it clearly!

### 3.2 Testing In-Process (Colab-Compatible)

In a real deployment, Claude Desktop or your app would launch the server as a subprocess. For Colab, we'll test in-process using FastMCP's built-in client:

In [ ]:
import asyncio
from mcp.server.fastmcp import FastMCP
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

# --- Define the server inline ---
mcp = FastMCP("MyFirstServer")

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together. Returns the sum."""
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    """Generate a friendly greeting for the given name."""
    return f"Hello, {name}! Welcome to MCP."

# --- Use FastMCP's in-process test client ---
async def test_server_inprocess():
    """Test tools without launching a subprocess"""
    async with mcp.test_client() as client:
        # List what tools are available
        tools = await client.list_tools()
        print("📋 Available Tools:")
        for tool in tools.tools:
            print(f"  • {tool.name}: {tool.description}")
            print(f"    Schema: {tool.inputSchema}")
        
        print()
        
        # Call add_numbers
        result = await client.call_tool("add_numbers", {"a": 15, "b": 27})
        print(f"🔧 add_numbers(15, 27) → {result.content[0].text}")
        
        # Call greet
        result = await client.call_tool("greet", {"name": "Gourav"})
        print(f"🔧 greet('Gourav') → {result.content[0].text}")

await test_server_inprocess()

### 💡 What Just Happened?

When you called `list_tools()`, the MCP protocol returned a JSON description of each tool — its **name**, **description**, and **input schema** (auto-generated from your type annotations). This is exactly what Claude sees when it decides which tool to call.

The schema for `add_numbers` looks like:
```json
{
  "type": "object",
  "properties": {
    "a": {"type": "integer"},
    "b": {"type": "integer"}
  },
  "required": ["a", "b"]
}
```

Claude reads this schema and knows exactly what arguments to provide.

---

## 📂 Part 4: MCP Resources — Read-Only Data

Resources are like REST API endpoints — they expose data for reading, not actions. They have a URI scheme like `file://path/to/doc` or `db://table/row`.

In [ ]:
from mcp.server.fastmcp import FastMCP
import json
from datetime import datetime

mcp_resources = FastMCP("ResourceServer")

# In-memory "database" of AI concepts
AI_CONCEPTS = {
    "rag": {
        "name": "Retrieval-Augmented Generation",
        "lesson": 7,
        "summary": "Augmenting LLMs with external knowledge retrieval to reduce hallucinations."
    },
    "mcp": {
        "name": "Model Context Protocol",
        "lesson": 14,
        "summary": "Standard protocol for connecting AI models to external tools and data sources."
    },
    "langgraph": {
        "name": "LangGraph",
        "lesson": 11,
        "summary": "Framework for building stateful, cyclical agent graphs with checkpointing."
    }
}

# --- RESOURCES: read-only data endpoints ---

@mcp_resources.resource("concepts://all")
def get_all_concepts() -> str:
    """Returns all AI concepts in the knowledge base"""
    return json.dumps(AI_CONCEPTS, indent=2)

@mcp_resources.resource("concepts://{concept_id}")
def get_concept(concept_id: str) -> str:
    """Returns a specific AI concept by ID"""
    if concept_id in AI_CONCEPTS:
        return json.dumps(AI_CONCEPTS[concept_id], indent=2)
    return json.dumps({"error": f"Concept '{concept_id}' not found"})

@mcp_resources.resource("system://timestamp")
def get_timestamp() -> str:
    """Returns the current UTC timestamp"""
    return datetime.utcnow().isoformat() + "Z"

# --- TOOLS that USE the resources internally ---

@mcp_resources.tool()
def lookup_concept(concept_id: str) -> str:
    """Look up an AI concept from the knowledge base by its ID (rag, mcp, langgraph)."""
    if concept_id.lower() in AI_CONCEPTS:
        c = AI_CONCEPTS[concept_id.lower()]
        return f"**{c['name']}** (Lesson {c['lesson']}): {c['summary']}"
    return f"Concept '{concept_id}' not found. Available: {', '.join(AI_CONCEPTS.keys())}"

# Test it!
async def test_resources():
    async with mcp_resources.test_client() as client:
        # List resources
        resources = await client.list_resources()
        print("📂 Available Resources:")
        for r in resources.resources:
            print(f"  • {r.uri}: {r.name}")
        
        print()
        
        # Read a resource
        content = await client.read_resource("concepts://mcp")
        print("📖 Reading concepts://mcp:")
        print(content.contents[0].text)
        
        print()
        
        # Call the tool
        result = await client.call_tool("lookup_concept", {"concept_id": "rag"})
        print(f"🔧 lookup_concept('rag') → {result.content[0].text}")

await test_resources()

### 📌 Tools vs. Resources — When to Use Which?

| | **Tools** | **Resources** |
|---|---|---|
| **Side effects** | ✅ Yes (can write, post, mutate) | ❌ No (read-only) |
| **Parameters** | Dynamic (passed at call time) | URI-template or static |
| **Use case** | Actions, computations, API calls | Data, files, config, state |
| **Example** | `send_email()`, `run_query()` | `file://config.json`, `db://users/42` |

In practice, **Tools are used ~80% of the time**. Resources are great for config, documents, and reference data.

---

## 💬 Part 5: MCP Prompts — Reusable Templates

Prompts are pre-built conversation starters. They're useful when you want to package a complex system prompt as part of your tool server.

In [ ]:
from mcp.server.fastmcp import FastMCP
from mcp.types import TextContent

mcp_prompts = FastMCP("PromptServer")

@mcp_prompts.prompt()
def code_reviewer(language: str, focus: str = "correctness") -> str:
    """A professional code review prompt for a specific language."""
    return f"""You are an expert {language} code reviewer with 15 years of experience.
Your focus is on {focus}. For every issue you find:
1. Cite the exact line/pattern
2. Explain WHY it's a problem
3. Provide a fixed version

Be direct, technical, and constructive. Do not praise code unless it's genuinely excellent."""

@mcp_prompts.prompt()
def research_assistant(topic: str, depth: str = "intermediate") -> str:
    """A research assistant prompt calibrated for depth level."""
    depth_map = {
        "beginner": "Use simple analogies, avoid jargon, focus on intuition.",
        "intermediate": "Use technical terms but define them. Balance depth with clarity.",
        "expert": "Go deep. Assume PhD-level background. Focus on nuance and edge cases."
    }
    guidance = depth_map.get(depth, depth_map["intermediate"])
    
    return f"""You are a research expert on {topic}.
{guidance}
Structure your responses with: Key Concepts → How it Works → Practical Applications → Open Questions."""

# Test prompts
async def test_prompts():
    async with mcp_prompts.test_client() as client:
        # List available prompts
        prompts = await client.list_prompts()
        print("💬 Available Prompts:")
        for p in prompts.prompts:
            print(f"  • {p.name}: {p.description}")
            if p.arguments:
                for arg in p.arguments:
                    req = "(required)" if arg.required else "(optional)"
                    print(f"    - {arg.name} {req}: {arg.description}")
        
        print()
        
        # Get a prompt
        result = await client.get_prompt("code_reviewer", {"language": "Python", "focus": "performance"})
        print("📝 Prompt 'code_reviewer' (Python, performance):")
        print(result.messages[0].content.text)

await test_prompts()

---

## 🤝 Part 6: Claude + MCP — Calling Tools via the API

Now the exciting part: making Claude actually *use* your MCP tools through the Anthropic API. The approach:

1. Start MCP server as a subprocess (stdio transport)
2. Connect an MCP client to it
3. Translate MCP tools → Anthropic tool definitions
4. Run the agent loop

This is the **bridge pattern** — connecting MCP servers to the Claude API manually.

In [ ]:
import json
import anthropic
from mcp.server.fastmcp import FastMCP

# ============================================================
# STEP 1: Define a practical MCP server with useful tools
# ============================================================

research_mcp = FastMCP("ResearchTools")

@research_mcp.tool()
def word_count(text: str) -> dict:
    """Count words, sentences, and characters in a piece of text.
    Returns a dict with counts and reading time estimate."""
    words = text.split()
    sentences = text.count('.') + text.count('!') + text.count('?')
    chars = len(text)
    reading_time_min = round(len(words) / 200, 1)  # ~200 wpm average
    return {
        "words": len(words),
        "sentences": max(sentences, 1),
        "characters": chars,
        "estimated_reading_time_minutes": reading_time_min
    }

@research_mcp.tool()
def extract_keywords(text: str, max_keywords: int = 5) -> list:
    """Extract the most frequent meaningful keywords from text.
    Filters common stop words. Returns up to max_keywords terms."""
    stop_words = {'the','a','an','is','are','was','were','be','been','being',
                  'have','has','had','do','does','did','will','would','could',
                  'should','may','might','shall','can','need','dare','ought',
                  'and','or','but','if','in','on','at','to','for','of','with',
                  'by','from','as','it','its','this','that','these','those',
                  'i','you','he','she','we','they','what','which','who','when',
                  'where','how','all','each','both','few','more','most','other',
                  'some','such','no','not','only','same','so','than','too','very'}
    
    words = [w.lower().strip('.,!?;:\'"()[]{}') for w in text.split()]
    freq = {}
    for word in words:
        if word and word not in stop_words and len(word) > 3:
            freq[word] = freq.get(word, 0) + 1
    
    sorted_keywords = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    return [kw for kw, _ in sorted_keywords[:max_keywords]]

@research_mcp.tool()
def summarize_structure(text: str) -> dict:
    """Analyze the structure of text: detects sections, bullet points, 
    numbered lists, and code blocks. Useful for understanding document layout."""
    lines = text.split('\n')
    structure = {
        "total_lines": len(lines),
        "headers": [l.strip() for l in lines if l.strip().startswith('#')],
        "bullet_points": sum(1 for l in lines if l.strip().startswith(('-', '*', '•'))),
        "numbered_items": sum(1 for l in lines if l.strip() and l.strip()[0].isdigit() and '. ' in l),
        "code_blocks": text.count('```'),
        "empty_lines": sum(1 for l in lines if not l.strip())
    }
    return structure

print("✅ ResearchTools MCP server defined with 3 tools")

In [ ]:
# ============================================================
# STEP 2: Bridge — convert MCP tools to Anthropic tool format
# ============================================================

def mcp_tool_to_anthropic(mcp_tool) -> dict:
    """Convert an MCP tool definition to Anthropic's tool format."""
    return {
        "name": mcp_tool.name,
        "description": mcp_tool.description,
        "input_schema": mcp_tool.inputSchema
    }

async def get_anthropic_tools_from_mcp(client):
    """Fetch tools from MCP server and convert to Anthropic format."""
    mcp_tools_response = await client.list_tools()
    return [mcp_tool_to_anthropic(t) for t in mcp_tools_response.tools]

# ============================================================
# STEP 3: Execute an MCP tool call (from Claude's response)
# ============================================================

async def execute_mcp_tool(mcp_client, tool_name: str, tool_input: dict) -> str:
    """Call an MCP tool and return its result as a string."""
    result = await mcp_client.call_tool(tool_name, tool_input)
    # MCP returns content blocks; get the text
    if result.content:
        content = result.content[0]
        if hasattr(content, 'text'):
            return content.text
    return str(result)

print("✅ Bridge functions defined")

In [ ]:
# ============================================================
# STEP 4: Full agent loop using Claude + MCP tools
# ============================================================

import anthropic

async def run_claude_with_mcp(user_message: str):
    """Run a complete Claude + MCP tool loop."""
    
    claude = anthropic.Anthropic()
    
    async with research_mcp.test_client() as mcp_client:
        # Get tools from MCP server, convert to Anthropic format
        anthropic_tools = await get_anthropic_tools_from_mcp(mcp_client)
        
        print(f"🔌 Connected to MCP server with {len(anthropic_tools)} tools:")
        for t in anthropic_tools:
            print(f"   • {t['name']}")
        print()
        print(f"👤 User: {user_message}")
        print()
        
        messages = [{"role": "user", "content": user_message}]
        
        # Agent loop
        while True:
            response = claude.messages.create(
                model="claude-opus-4-6",
                max_tokens=1024,
                system="You are a helpful research assistant. Use your tools when analyzing text.",
                tools=anthropic_tools,
                messages=messages
            )
            
            # Check if Claude wants to use tools
            if response.stop_reason == "tool_use":
                # Extract tool calls
                tool_calls = [b for b in response.content if b.type == "tool_use"]
                
                # Execute each tool via MCP
                tool_results = []
                for tc in tool_calls:
                    print(f"🔧 Claude calling tool: {tc.name}({tc.input})")
                    result = await execute_mcp_tool(mcp_client, tc.name, tc.input)
                    print(f"   → Result: {result[:200]}..." if len(str(result)) > 200 else f"   → Result: {result}")
                    print()
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tc.id,
                        "content": str(result)
                    })
                
                # Add assistant response + tool results to conversation
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": tool_results})
                
            else:
                # Claude is done — extract final text
                final_text = next(
                    (b.text for b in response.content if hasattr(b, 'text')), 
                    "No text response"
                )
                print(f"🤖 Claude: {final_text}")
                break

# Test with a real request
sample_text = """
Model Context Protocol (MCP) is an open standard for connecting AI systems to tools and data.
It defines three primitives: tools for actions, resources for data, and prompts for templates.
MCP uses JSON-RPC over stdio or HTTP. Any AI model can use any MCP server without custom code.
This makes AI systems more modular, composable, and maintainable.
"""

await run_claude_with_mcp(
    f"Analyze this text and give me: word count stats, top keywords, and structure info.\n\nText: {sample_text}"
)

### 🧠 What Just Happened?

1. **Claude received** the user message + a list of available MCP tools (auto-converted from MCP schema → Anthropic tool format)
2. **Claude decided** which tools to call and with what arguments (like a function signature)
3. **Your code** relayed those calls to the MCP server and got results back
4. **Claude synthesized** the results into a natural language response

The key insight: **Claude never executes code itself**. It just outputs structured JSON describing *what to call*. Your bridge code does the actual execution. This is the same pattern at the core of all AI agents.

---

## 🏭 Part 7: Production MCP Server — Deployment Patterns

### 7.1 Running as a Real Subprocess (Local stdio)

In production (Claude Desktop, Cowork, your custom app), the server runs as a subprocess:

In [ ]:
# This is what a production MCP server file looks like
# Save this as research_mcp_server.py and run it directly

PRODUCTION_SERVER = '''
#!/usr/bin/env python3
"""
research_mcp_server.py
A production-ready MCP server for research assistance.
Run: python research_mcp_server.py
Or register in Claude Desktop config.
"""

from mcp.server.fastmcp import FastMCP
import json
import os

mcp = FastMCP(
    name="ResearchAssistant",
    # Optional: add dependencies that get auto-installed
    dependencies=["anthropic", "httpx"]
)

@mcp.tool()
def word_count(text: str) -> dict:
    """Count words, sentences, and characters. Returns reading time estimate."""
    words = text.split()
    return {
        "words": len(words),
        "characters": len(text),
        "reading_time_minutes": round(len(words) / 200, 1)
    }

@mcp.tool()
def save_note(title: str, content: str, tags: list[str] = None) -> str:
    """Save a research note to disk. Returns the file path."""
    notes_dir = os.path.expanduser("~/research_notes")
    os.makedirs(notes_dir, exist_ok=True)
    
    filename = title.lower().replace(" ", "_") + ".json"
    filepath = os.path.join(notes_dir, filename)
    
    note = {"title": title, "content": content, "tags": tags or []}
    with open(filepath, "w") as f:
        json.dump(note, f, indent=2)
    
    return f"Note saved to {filepath}"

@mcp.resource("notes://list")
def list_notes() -> str:
    """List all saved research notes"""
    notes_dir = os.path.expanduser("~/research_notes")
    if not os.path.exists(notes_dir):
        return json.dumps([])
    files = [f for f in os.listdir(notes_dir) if f.endswith(".json")]
    return json.dumps(files)

if __name__ == "__main__":
    # Runs with stdio transport by default — perfect for Claude Desktop/Cowork
    mcp.run()
'''

# Save to disk
with open("/tmp/research_mcp_server.py", "w") as f:
    f.write(PRODUCTION_SERVER.strip())

print("✅ Production server saved to /tmp/research_mcp_server.py")
print()
print("📋 Claude Desktop config to register this server:")
config = {
    "mcpServers": {
        "research-assistant": {
            "command": "python",
            "args": ["/path/to/research_mcp_server.py"]
        }
    }
}
print(json.dumps(config, indent=2))
print()
print("This goes in: ~/Library/Application Support/Claude/claude_desktop_config.json (macOS)")

### 7.2 HTTP/SSE Transport (Remote Servers)

For servers that need to be accessible remotely (shared team tool, deployed service):

In [ ]:
HTTP_SERVER_CODE = '''
# Run with: python server.py
# Accessible at http://localhost:8000/sse

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("RemoteResearchServer")

@mcp.tool()
def ping() -> str:
    """Simple health check tool."""
    return "pong"

if __name__ == "__main__":
    # HTTP transport — accessible over the network
    mcp.run(transport="sse", host="0.0.0.0", port=8000)
    # OR the newer streamable-http:
    # mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)
'''

print("HTTP MCP Server pattern:")
print(HTTP_SERVER_CODE)

print("\n📋 Client connection for HTTP transport:")
CLIENT_CODE = '''
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

async with sse_client("http://localhost:8000/sse") as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        print(tools)
'''
print(CLIENT_CODE)
print("💡 EXPERIMENT: Deploy your MCP server to a cloud VM and connect from anywhere!")

---

## 🏆 Capstone: Build the AutoResearcher MCP Server

Remember the AutoResearcher from Lesson 9? Let's give it an MCP-powered interface — turning it into a pluggable tool server any AI can use.

**Goal:** Build an MCP server with tools that could plug into Claude Desktop, Cowork, or any MCP-compatible host.

In [ ]:
from mcp.server.fastmcp import FastMCP
import anthropic
import json
import time
from datetime import datetime
from typing import Optional

# ============================================================
# AutoResearcher MCP Server
# A self-contained research assistant as an MCP server
# ============================================================

auto_researcher = FastMCP(
    name="AutoResearcher",
    instructions="A research assistant that helps analyze topics, outline arguments, and evaluate evidence."
)

# In-memory session store (in production: use Redis or a DB)
_research_sessions = {}

# -------- Tool 1: Start a research session --------
@auto_researcher.tool()
def start_research_session(topic: str, depth: str = "intermediate") -> dict:
    """Start a new research session on a topic.
    depth: 'quick' (overview), 'intermediate' (detailed), 'deep' (exhaustive).
    Returns a session_id to use with other tools."""
    
    session_id = f"research_{int(time.time())}"
    _research_sessions[session_id] = {
        "topic": topic,
        "depth": depth,
        "started_at": datetime.utcnow().isoformat(),
        "findings": [],
        "status": "active"
    }
    
    return {
        "session_id": session_id,
        "topic": topic,
        "depth": depth,
        "message": f"Research session started. Use session_id '{session_id}' with other tools."
    }

# -------- Tool 2: Generate a research outline --------
@auto_researcher.tool()
def generate_research_outline(session_id: str) -> str:
    """Generate a structured research outline for the topic in the given session.
    Uses Claude to create section headings and key questions to investigate."""
    
    if session_id not in _research_sessions:
        return json.dumps({"error": f"Session '{session_id}' not found."})
    
    session = _research_sessions[session_id]
    topic = session["topic"]
    depth = session["depth"]
    
    claude = anthropic.Anthropic()
    
    depth_guidance = {
        "quick": "3 sections, 2 questions each",
        "intermediate": "5 sections, 3 questions each",
        "deep": "7 sections, 4 questions each"
    }.get(depth, "5 sections, 3 questions each")
    
    response = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": 
            f"Create a research outline for: {topic}\n"
            f"Format: {depth_guidance}\n"
            f"Output: numbered sections with bullet-point questions. Be specific and investigative."
        }]
    )
    
    outline = response.content[0].text
    session["outline"] = outline
    
    return outline

# -------- Tool 3: Add a research finding --------
@auto_researcher.tool()
def add_finding(
    session_id: str, 
    finding: str, 
    source: str = "manual",
    confidence: str = "medium"
) -> dict:
    """Add a research finding to the session.
    confidence: 'high', 'medium', or 'low'.
    Returns updated finding count."""
    
    if session_id not in _research_sessions:
        return {"error": f"Session '{session_id}' not found."}
    
    session = _research_sessions[session_id]
    entry = {
        "finding": finding,
        "source": source,
        "confidence": confidence,
        "added_at": datetime.utcnow().isoformat()
    }
    session["findings"].append(entry)
    
    return {
        "status": "added",
        "total_findings": len(session["findings"]),
        "finding_preview": finding[:100] + "..." if len(finding) > 100 else finding
    }

# -------- Tool 4: Synthesize findings into a report --------
@auto_researcher.tool()
def synthesize_report(session_id: str, format: str = "markdown") -> str:
    """Synthesize all findings in a session into a cohesive research report.
    format: 'markdown', 'json', or 'bullet_points'.
    Returns the complete report."""
    
    if session_id not in _research_sessions:
        return f"Error: Session '{session_id}' not found."
    
    session = _research_sessions[session_id]
    
    if not session["findings"]:
        return "No findings yet. Use add_finding() to add research findings first."
    
    claude = anthropic.Anthropic()
    
    findings_text = "\n".join([
        f"- [{f['confidence'].upper()} confidence] {f['finding']} (source: {f['source']})"
        for f in session["findings"]
    ])
    
    response = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": 
            f"Research Topic: {session['topic']}\n\n"
            f"Findings:\n{findings_text}\n\n"
            f"Synthesize these findings into a coherent research report in {format} format. "
            f"Highlight high-confidence findings. Note areas of uncertainty. "
            f"End with 3 actionable conclusions."
        }]
    )
    
    report = response.content[0].text
    session["report"] = report
    session["status"] = "completed"
    
    return report

# -------- Resource: View active sessions --------
@auto_researcher.resource("research://sessions")
def list_sessions() -> str:
    """List all active research sessions"""
    summary = {
        sid: {
            "topic": s["topic"],
            "status": s["status"],
            "findings_count": len(s["findings"]),
            "started_at": s["started_at"]
        }
        for sid, s in _research_sessions.items()
    }
    return json.dumps(summary, indent=2)

print("✅ AutoResearcher MCP Server defined!")
print("   Tools: start_research_session, generate_research_outline, add_finding, synthesize_report")
print("   Resources: research://sessions")

In [ ]:
# ============================================================
# Test the full AutoResearcher MCP workflow
# ============================================================

async def run_full_research_workflow():
    """Demonstrate the full research workflow via MCP"""
    
    async with auto_researcher.test_client() as client:
        print("=" * 60)
        print("🔬 AutoResearcher MCP — Full Workflow Demo")
        print("=" * 60)
        print()
        
        # Step 1: List available tools
        tools = await client.list_tools()
        print(f"📋 Available Tools ({len(tools.tools)} total):")
        for t in tools.tools:
            print(f"   • {t.name}: {t.description[:80]}...")
        print()
        
        # Step 2: Start a research session
        print("🚀 Step 1: Starting research session...")
        result = await client.call_tool("start_research_session", {
            "topic": "The impact of MCP on AI agent architectures",
            "depth": "intermediate"
        })
        session_data = json.loads(result.content[0].text)
        session_id = session_data["session_id"]
        print(f"   Session ID: {session_id}")
        print()
        
        # Step 3: Generate outline
        print("📋 Step 2: Generating research outline...")
        result = await client.call_tool("generate_research_outline", {"session_id": session_id})
        print(result.content[0].text)
        print()
        
        # Step 4: Add findings
        print("📝 Step 3: Adding research findings...")
        findings_to_add = [
            ("MCP reduces N×M integration complexity to N+M by standardizing the tool interface", "mcp.io docs", "high"),
            ("FastMCP enables building MCP servers in Python with minimal boilerplate using decorators", "this lesson", "high"),
            ("Adoption by OpenAI and Google suggests MCP will become the de-facto standard", "industry reports", "medium"),
            ("stdio transport has lower latency than HTTP for local tools", "benchmark tests", "medium"),
        ]
        
        for finding, source, confidence in findings_to_add:
            result = await client.call_tool("add_finding", {
                "session_id": session_id,
                "finding": finding,
                "source": source,
                "confidence": confidence
            })
            data = json.loads(result.content[0].text)
            print(f"   ✓ Finding #{data['total_findings']} added")
        print()
        
        # Step 5: Synthesize report
        print("📊 Step 4: Synthesizing final report...")
        print("-" * 60)
        result = await client.call_tool("synthesize_report", {
            "session_id": session_id,
            "format": "markdown"
        })
        print(result.content[0].text)
        print("-" * 60)
        print()
        
        # Step 6: Check resource
        print("📂 Step 5: Checking sessions resource...")
        resource_content = await client.read_resource("research://sessions")
        sessions_data = json.loads(resource_content.contents[0].text)
        for sid, info in sessions_data.items():
            print(f"   Session: {sid}")
            print(f"   Topic: {info['topic']}")
            print(f"   Status: {info['status']} | Findings: {info['findings_count']}")
        
        print()
        print("✅ Full MCP workflow complete!")

await run_full_research_workflow()

---

## 🎯 Key Takeaways

You've just built and tested a full MCP server. Here's what to internalize:

**1. MCP = USB-C for AI**  
One standard that decouples AI models from tools. Write a tool once as MCP → works with Claude, GPT, Gemini, and future models.

**2. FastMCP makes it trivial**  
Three decorators: `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`. Type annotations become JSON schema automatically.

**3. The docstring is your prompt**  
Claude reads your tool's docstring to decide when and how to use it. Write it like you're explaining to a smart colleague.

**4. The bridge pattern**  
MCP tools → Anthropic tool format → agent loop. This is how all production MCP-enabled Claude apps work under the hood.

**5. You've been using MCP all along**  
Every time Claude in Cowork reads your files, searches the web, or creates calendar events — it's calling MCP tools exactly like you built here.

---

## 💡 Experiments to Try

1. **Add a web search tool** — use `httpx` to call DuckDuckGo's API in a new `@mcp.tool()`
2. **Persist sessions to disk** — replace `_research_sessions` dict with JSON files
3. **Add error handling** — what happens when Claude passes wrong argument types?
4. **Build an HTTP server** — run `mcp.run(transport="sse", port=8000)` and connect via HTTP client
5. **Register in Claude Desktop** — if you have Claude Desktop installed, add `research_mcp_server.py` to your config and chat with Claude about it!

---

## 🗺️ What's Next

**Lesson 15 (Final): Open-Source Project Publishing**  
You've built: agents, RAG, multi-agent systems, LangGraph, fine-tuning, vision AI, and now MCP servers. In the final lesson, we'll package your AutoResearcher as a real open-source project on GitHub — complete with README, CI, tests, and community-ready structure. This is how you demonstrate your expertise to the world. 🚀

In [ ]:
# ============================================================
# BONUS: MCP Inspector — Pretty-print all server capabilities
# ============================================================

async def inspect_mcp_server(server: FastMCP):
    """Print a detailed overview of all MCP server capabilities"""
    
    async with server.test_client() as client:
        print(f"🔍 MCP Server Inspector")
        print(f"{'='*50}")
        print(f"Server: {server.name}")
        print()
        
        # Tools
        tools_resp = await client.list_tools()
        print(f"🔧 TOOLS ({len(tools_resp.tools)}):")
        for tool in tools_resp.tools:
            print(f"  ┌─ {tool.name}")
            print(f"  │  Description: {tool.description}")
            props = tool.inputSchema.get('properties', {})
            required = tool.inputSchema.get('required', [])
            if props:
                print(f"  │  Parameters:")
                for param, schema in props.items():
                    req_marker = "*" if param in required else "?"
                    ptype = schema.get('type', schema.get('anyOf', [{}])[0].get('type', 'any'))
                    default = f" = {schema.get('default')}" if 'default' in schema else ""
                    print(f"  │    {req_marker} {param}: {ptype}{default}")
            print(f"  └─")
        print()
        
        # Resources
        try:
            resources_resp = await client.list_resources()
            if resources_resp.resources:
                print(f"📂 RESOURCES ({len(resources_resp.resources)}):")
                for r in resources_resp.resources:
                    print(f"  • {r.uri} — {r.name}")
                print()
        except:
            pass
        
        # Prompts
        try:
            prompts_resp = await client.list_prompts()
            if prompts_resp.prompts:
                print(f"💬 PROMPTS ({len(prompts_resp.prompts)}):")
                for p in prompts_resp.prompts:
                    print(f"  • {p.name}: {p.description}")
                print()
        except:
            pass

# Inspect the AutoResearcher server
await inspect_mcp_server(auto_researcher)

print("\n💡 EXPERIMENT: Call inspect_mcp_server(research_mcp) to inspect the ResearchTools server too!")